# Simulation: the firmware on the laptop
The ESP32 isn't connected yet, so the firmware runs in the MicroPython Unix port first. The code in `firmware/` is exactly the code that will be copied to the board. Only `emlearn_fft` is replaced by a copy of its FFT in Python (`sim/run_sim.py`), because emlearn has no build for this Mac.

## Day 8 – the 12 features in MicroPython
`firmware/features.py` computes the same 12 features as `src/features.py`, but sample by sample from a ring buffer, with pre-allocated `array` buffers and no numpy. Today I check that both give the same numbers.

### Setup
`run_firmware(x, op)` writes the signal to a file, starts MicroPython and feeds the samples one by one into a ring buffer of 1024 samples per axis, the way they arrive on the board. Every 256 samples it computes one window, so it returns one row of 12 features per window, like `features(x, op)`.

In [1]:
import sys
sys.path.insert(0, '..')
import time
import numpy as np, pandas as pd
from sim.run_sim import run_firmware, run_scores
from src.features import FREQ, NAMES, SPEED, WIN, features, index_recordings, load

idx = index_recordings()

### The spindle band as FFT bins
On the laptop the band is every FFT bin within ±10% of the spindle speed, found with a float comparison. The board uses integer maths instead: the first bin at or above 0.9 × speed and the last bin at or below 1.1 × speed. This cell prints the laptop bins, which should be 93–112 for 200 Hz and 116–140 for 250 Hz, the values the integer formula in `spindle_bins` gives.

In [2]:
for s in sorted(set(SPEED.values())):
    near = np.flatnonzero(np.abs(FREQ - s) <= .1 * s)
    lo, hi = (9 * s * WIN + 10 * 2000 - 1) // (10 * 2000), 11 * s * WIN // (10 * 2000)
    print(f'{s} Hz: laptop bins {near[0]}-{near[-1]}, firmware formula {lo}-{hi}')

200 Hz: laptop bins 93-112, firmware formula 93-112
250 Hz: laptop bins 116-140, firmware formula 116-140


### Test signals
- **Sine:** 250 Hz with amplitude 100 counts and an offset of -1000 counts, where the answer is known: RMS 70.7, peak-to-peak 200, crest factor 1.414.
- **Flat:** all zeros. RMS is 0, so the crest factor divides by EPS only, and the spindle energy is log10 of EPS only.
- **Real recordings:** complete good and bad recordings of OP01 (250 Hz) and OP05 (200 Hz), so both band widths are tested.

In [3]:
t = np.arange(2000 * 5) / 2000
sine = np.tile(100 * np.sin(2 * np.pi * 250 * t)[:, None] - 1000, (1, 3))
cases = {'sine 250 Hz': (sine, 'OP01'), 'flat': (np.zeros((WIN + 4 * 256, 3)), 'OP01')}
for op in ['OP01', 'OP05']:
    for lab in ['good', 'bad']:
        r = idx[(idx['op'] == op) & (idx['label'] == lab)].iloc[0]
        cases[f"{op} {lab} {r['period']} {r['num']}"] = (load(r['path']), op)
for name, (x, op) in cases.items():
    print(f'{name:24} {len(x) / 2000:5.1f} s')

sine 250 Hz                5.0 s
flat                       1.0 s
OP01 good 2019-08 000     30.2 s
OP01 bad 2019-08 000      19.5 s
OP05 good 2019-08 000     23.0 s
OP05 bad 2019-08 000      13.4 s


### Firmware against laptop
For every test signal: features from MicroPython and from numpy, then the largest absolute difference per feature, over all windows. RMS and peak-to-peak are in counts, crest factor has no unit and the spindle energy is in dB.

In [4]:
gaps, n = [], 0
start = time.time()
for name, (x, op) in cases.items():
    fw, lap = run_firmware(x, op), features(x, op)
    assert fw.shape == lap.shape, name
    gaps.append(pd.Series(np.abs(fw - lap).max(axis=0), NAMES, name=name))
    n += len(fw)
gaps = pd.DataFrame(gaps)
print(f'{n} windows in {time.time() - start:.0f} s')
print(gaps.T.to_string(float_format='{:.1e}'.format))

701 windows in 11 s
              sine 250 Hz    flat  OP01 good 2019-08 000  OP01 bad 2019-08 000  OP05 good 2019-08 000  OP05 bad 2019-08 000
rms X             7.6e-06 0.0e+00                3.0e-05               3.0e-05                2.9e-05               1.5e-05
rms Y             7.6e-06 0.0e+00                1.4e-05               1.3e-05                1.2e-05               7.5e-06
rms Z             7.6e-06 0.0e+00                1.5e-05               1.3e-05                7.6e-06               7.5e-06
p2p X             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
p2p Y             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
p2p Z             0.0e+00 0.0e+00                0.0e+00               0.0e+00                0.0e+00               0.0e+00
crest X           1.2e-07 0.0e+00                4.2e-07               2.3e-07                8.3e-07           

### Sine check
The firmware values of the first sine window against the expected values.

In [5]:
fw = run_firmware(sine, 'OP01')
print(pd.Series(fw[0], NAMES).round(4).to_string())

rms X            70.710701
rms Y            70.710701
rms Z            70.710701
p2p X           200.000000
p2p Y           200.000000
p2p Z           200.000000
crest X           1.414200
crest Y           1.414200
crest Z           1.414200
spindle_db X     89.921501
spindle_db Y     89.921501
spindle_db Z     89.921501


### What Day 8 shows
- **The firmware gives the laptop's numbers.** Over 701 windows of six signals, the largest difference is 3.0e-05 counts for RMS, 0 for peak-to-peak, 8.3e-07 for the crest factor and 6.7e-06 dB for the spindle energy. These differences come from storing the results as float32. The model's threshold sits several dB above normal, so differences this small can't change an alarm.
- **The spindle band is the same.** The integer formula on the board picks bins 93–112 for 200 Hz and 116–140 for 250 Hz, the same bins as the laptop.
- **The sine gives the expected values:** RMS 70.71, peak-to-peak 200 and crest factor 1.414, with the -1000 offset removed.
- **Limits of this check:** the Mac's MicroPython calculates in double precision, while the ESP32-S3 uses single precision, so the board may differ a little more. The FFT here is the Python copy of emlearn's FFT, not emlearn itself. Both have to be checked again on the board, and the time on the laptop says nothing about the time on the chip.

## Day 9 – the Mahalanobis score in MicroPython
Day 8 gave the 12 features on the board. Today the board also gets the model: the Mahalanobis distance from the period centre, divided by the operation's threshold, so 1 is the threshold again. `tools/stream.py` will send the operation and the period, so the board loads the matching centre. The 5 s median, the idle filter and the 3 s LED hold come later with `firmware/main.py`.

### Setup
The model functions from notebook 03 (Days 5 and 6) now live in `src/models.py`: the split by operation and period, the fit with a centre per period, the score and the threshold. This cell computes the features of all 447 recordings again (about 20 s) and makes the same split as notebook 03.

In [6]:
from src.models import fit, score, split_by_period, threshold

F = {i: features(load(p), op) for i, (p, op) in enumerate(zip(idx['path'], idx['op']))}
idx['split'] = split_by_period(idx)
print(idx.groupby(['split', 'label']).size().unstack(fill_value=0))

label  bad  good
split           
test    34    81
train    0   244
val      0    88


### Refit and check against notebook 03
One model per operation, fitted on the train recordings, threshold from the validation recordings. If the code moved to `src/` correctly, the thresholds are the ones printed on Day 6 of notebook 03.

In [7]:
DAY6 = {'OP01': 3.95, 'OP02': 3.48, 'OP03': 4.15, 'OP04': 4.52, 'OP05': 3.26, 'OP06': 5.14, 'OP07': 3.54,
        'OP08': 3.9, 'OP09': 4.89, 'OP10': 3.58, 'OP11': 4.88, 'OP12': 3.89, 'OP14': 3.7}  # notebook 03, Day 6
M = {}
for op, g in idx.groupby('op'):
    M[op] = fit(F, g[g['split'] == 'train'])
    M[op]['thr'] = threshold(M[op], F, g[g['split'] == 'val'])
thr = pd.Series({op: m['thr'] for op, m in M.items()})
print(pd.DataFrame({'now': thr.round(2), 'day 6': pd.Series(DAY6)}).T.to_string())
assert (thr.round(2) == pd.Series(DAY6)).all()

       OP01  OP02  OP03  OP04  OP05  OP06  OP07  OP08  OP09  OP10  OP11  OP12  OP14
now    3.95  3.48  4.15  4.52  3.26  5.14  3.54   3.9  4.89  3.58  4.88  3.89   3.7
day 6  3.95  3.48  4.15  4.52  3.26  5.14  3.54   3.9  4.89  3.58  4.88  3.89   3.7


### Export firmware/model.bin
One record for every operation and period that has a centre. Each record is a 12-byte key such as `OP01 2019-08`, then 169 float32 numbers: the centre (12), the std (12), the inverse covariance (12 × 12) and the threshold (1). Every record has the same size, so the board can read them one after another into one buffer and stop at the key it was sent. float32 is the number format of the board, so rounding to it happens here, once.

In [8]:
rec = []
for op, m in M.items():
    for period, c in m['centre'].items():
        key = f'{op} {period}'.encode()
        assert len(key) == 12
        rec.append(key + np.r_[c, m['std'], m['inv'].ravel(), m['thr']].astype('<f4').tobytes())
with open('../firmware/model.bin', 'wb') as f:
    f.write(b''.join(rec))
print(len(rec), 'records of', len(rec[0]), 'bytes =', len(rec) * len(rec[0]), 'bytes')

59 records of 688 bytes = 40592 bytes


### Firmware score against laptop score
For every operation, one good and one bad test recording, both from a period with a centre. MicroPython computes features and score sample by sample (`run_scores`), numpy computes `score / threshold` from the laptop features. The firmware scores every window, so idle windows are compared too. Besides the largest difference, the cell counts windows where the two disagree about being above the threshold (score > 1).

In [9]:
test = idx[idx['split'] == 'test']
rows, start = [], time.time()
for op, g in test.groupby('op'):
    for lab in ['good', 'bad']:
        i = next(i for i in g.index[g['label'] == lab] if g.loc[i, 'period'] in M[op]['centre'])
        x, period = load(idx.loc[i, 'path']), idx.loc[i, 'period']
        fw = run_scores(x, op, period)
        lap = score(M[op], features(x, op), period) / M[op]['thr']
        rows.append(dict(op=op, label=lab, period=period, windows=len(fw), max_score=lap.max(),
                         max_diff=np.abs(fw - lap).max(), max_rel_diff=(np.abs(fw - lap) / lap).max(),
                         disagree=((fw > 1) != (lap > 1)).sum()))
cmp = pd.DataFrame(rows)
print(f'{cmp.windows.sum()} windows in {time.time() - start:.0f} s')
print(cmp.to_string(index=False, float_format='{:.2e}'.format))

9769 windows in 164 s
  op label  period  windows  max_score  max_diff  max_rel_diff  disagree
OP01  good 2019-08      229   3.34e+00  5.33e-07      1.39e-06         0
OP01   bad 2019-08      149   1.62e+00  3.43e-07      3.53e-07         0
OP02  good 2019-08      341   2.57e+00  7.75e-07      8.70e-07         0
OP02   bad 2019-02      277   5.09e+00  1.08e-06      3.79e-07         0
OP03  good 2019-08      693   4.78e+00  7.15e-07      1.21e-06         0
OP03   bad 2019-08      542   1.90e+00  5.16e-07      6.56e-07         0
OP04  good 2019-08      533   3.06e+00  6.07e-07      1.23e-06         0
OP04   bad 2019-08      261   2.77e+00  8.66e-07      5.87e-07         0
OP05  good 2019-08      177   2.57e+00  5.44e-07      6.54e-07         0
OP05   bad 2019-08      101   2.45e+00  5.10e-07      4.53e-07         0
OP06  good 2019-08      621   3.84e+00  3.86e-07      9.70e-07         0
OP06   bad 2019-08      661   1.40e+00  5.35e-07      9.07e-07         0
OP07  good 2019-08      193  

### What Day 9 shows
- **The move to `src/models.py` changed nothing.** The refit gives the same split and the same 13 thresholds as Day 6 of notebook 03, from 3.26 (OP05) to 5.14 (OP06).
- **The model is small.** `model.bin` has 59 records (one per operation and period with a centre) of 688 bytes, 40,592 bytes in total. The board only ever holds one record in RAM: 169 float32 numbers, 676 bytes.
- **The firmware gives the laptop's score.** Over 9,769 windows of 26 test recordings, the largest difference is 1.1e-06 on the score / threshold scale, at most 2.0e-06 relative to the score. No window is above the threshold on one side and below it on the other.
- **These are single-window scores.** Good recordings go above 1 here too, because the threshold was set on the 5 s median score, not on single windows. The median, the idle filter and the 3 s LED hold come with `firmware/main.py`.
- **Limits of this check:** the same as Day 8. The Mac's MicroPython calculates in double precision and the ESP32-S3 in single precision, and OP10 2021-08 has no record, so the board can't score it.